# DyTopo — Embedding Analysis

**Why does the mock data produce a dense→sparse topology evolution?**

The `_CODE_MOCK` / `_MATH_MOCK` dicts in `dytopo_engine.py` contain hand-crafted Q/K descriptor strings.
The strings were written so that when encoded by `all-MiniLM-L6-v2` and compared via cosine similarity,
the resulting similarity matrix R = Q @ K.T produces the right density at threshold τ=0.3.

This notebook verifies and visualises why:
1. **Round 1 Q/K pairs** are broad/overlapping → high cosine similarities → many edges above τ=0.3 → **dense graph**
2. **Round 2 Q/K pairs** are role-specific → moderate overlap → **medium density**
3. **Round 3 Q/K pairs** signal completion → low overlap → few edges above τ=0.3 → **sparse graph**

This is the paper's Figure 4 pattern: *broad exploratory routing → verification-focused → dependency-minimal*.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

API_DIR = os.path.join(os.path.dirname(os.path.abspath('')), 'api')
if API_DIR not in sys.path:
    sys.path.insert(0, API_DIR)

from dytopo_engine import encode, build_adjacency, _CODE_MOCK, _MATH_MOCK, CODE_ROLES, MATH_ROLES
print('Imports OK — loading sentence encoder (first run downloads ~90MB)...')

# Trigger encoder load
_ = encode(['test'])
print('Encoder loaded.')

## 1. Extract Q/K vectors from mock data per round

In [ ]:
def extract_qk(mock_dict, roles, n_rounds=3):
    """Extract ordered Q and K descriptor strings per round."""
    result = []
    for r in range(n_rounds):
        qs, ks = [], []
        for role in roles:
            entry = mock_dict.get((role, r))
            if entry:
                qs.append(entry['q_vector'])
                ks.append(entry['k_vector'])
            else:
                qs.append(f'I need information relevant to {role}')
                ks.append(f'I provide {role.lower()} expertise')
        result.append((qs, ks))
    return result

code_qk = extract_qk(_CODE_MOCK, CODE_ROLES, n_rounds=3)
math_qk = extract_qk(_MATH_MOCK, MATH_ROLES[:3], n_rounds=3)  # ProblemParser, Solver, Verifier

for r, (qs, ks) in enumerate(code_qk):
    print(f'=== Code Round {r+1} ===')
    for role, q, k in zip(CODE_ROLES, qs, ks):
        print(f'  {role:12s}  Q: {q[:55]}...' if len(q) > 55 else f'  {role:12s}  Q: {q}')
        print(f'               K: {k[:55]}...' if len(k) > 55 else f'               K: {k}')
    print()

## 2. Compute similarity matrices R = Q @ K.T per round

In [ ]:
TAU = 0.3
code_R = []
code_A = []

for r, (qs, ks) in enumerate(code_qk):
    Q = encode(qs)
    K = encode(ks)
    R = Q @ K.T
    A = build_adjacency(qs, ks, TAU)
    code_R.append(R)
    code_A.append(A)
    density = A.sum() / (len(qs) * (len(qs) - 1))
    print(f'Code Round {r+1}: edges above τ={TAU} → {int(A.sum())} edges, density={density:.0%}')
    print(f'  R min={R.min():.3f}  max={R.max():.3f}  mean (off-diag)={R[~np.eye(len(qs),dtype=bool)].mean():.3f}')

## 3. Visualise similarity matrices per round

In [ ]:
N = len(CODE_ROLES)
PHASES = ['Round 1\n(Exploration)', 'Round 2\n(Verification)', 'Round 3\n(Finalization)']

fig, axes = plt.subplots(2, 3, figsize=(13, 9))

for r in range(3):
    # Top row: raw cosine similarity R
    ax = axes[0, r]
    R = code_R[r].copy()
    np.fill_diagonal(R, 0)
    density = code_A[r].sum() / (N * (N - 1))

    im = ax.imshow(R, cmap='RdYlGn', vmin=-0.2, vmax=0.8, aspect='equal')
    ax.set_xticks(range(N))
    ax.set_yticks(range(N))
    ax.set_xticklabels(CODE_ROLES, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(CODE_ROLES, fontsize=8)
    ax.set_title(f'{PHASES[r]}\nCosine Similarity R', fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)

    for i in range(N):
        for j in range(N):
            v = R[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if abs(v) > 0.4 else 'black')

    # Bottom row: binary adjacency A (edges above tau)
    ax2 = axes[1, r]
    A = code_A[r]
    cmap_bin = plt.cm.Blues
    ax2.imshow(A, cmap=cmap_bin, vmin=0, vmax=1, aspect='equal')
    ax2.set_xticks(range(N))
    ax2.set_yticks(range(N))
    ax2.set_xticklabels(CODE_ROLES, rotation=45, ha='right', fontsize=8)
    ax2.set_yticklabels(CODE_ROLES, fontsize=8)
    ax2.set_title(f'Binary Adjacency A (τ={TAU})\nDensity={density:.0%}', fontsize=9)

    for i in range(N):
        for j in range(N):
            ax2.text(j, i, str(A[i, j]), ha='center', va='center', fontsize=10,
                     color='white' if A[i, j] else '#475569', fontweight='bold')

axes[0, 0].set_ylabel('Query agent (needs)', fontsize=9, color='white')
axes[1, 0].set_ylabel('Destination agent', fontsize=9, color='white')

for ax in axes.flat:
    ax.set_facecolor('#0f1828')
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#1e293b')

fig.patch.set_facecolor('#0a0e1a')
fig.suptitle('Mock Q/K Cosine Similarities → Adjacency Matrices (Code Task, τ=0.3)\n'
             'Why mock data produces Figure 4 topology evolution',
             fontsize=11, color='white', fontweight='bold')
plt.tight_layout()
plt.savefig('dytopo_embedding_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved: dytopo_embedding_analysis.png')

## 4. Q/K descriptor similarity distribution per round

Shows the *distribution* of off-diagonal R values shifting rightward (higher similarity) in Round 1 vs. Round 3.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

COLORS = ['#38bdf8', '#fbbf24', '#4ade80']
LABELS = ['Round 1 (Exploration)', 'Round 2 (Verification)', 'Round 3 (Finalization)']

mask = ~np.eye(N, dtype=bool)

for r, (R, col, lbl) in enumerate(zip(code_R, COLORS, LABELS)):
    vals = R[mask]
    ax.hist(vals, bins=15, alpha=0.55, color=col, label=lbl, edgecolor='none')
    ax.axvline(np.mean(vals), color=col, linestyle='--', linewidth=2, alpha=0.9)

ax.axvline(TAU, color='white', linestyle='-', linewidth=2, alpha=0.7, label=f'τ={TAU} threshold')
ax.text(TAU + 0.01, ax.get_ylim()[1] * 0.9 if ax.get_ylim()[1] > 0 else 1,
        f'τ={TAU}', color='white', fontsize=9)

ax.set_xlabel('Cosine Similarity (Q_i · K_j)', fontsize=11, color='white')
ax.set_ylabel('Count', fontsize=11, color='white')
ax.set_title('Off-Diagonal Cosine Similarity Distribution per Round\n'
             'Dashed lines = mean · Values right of τ threshold → active edges',
             fontsize=10, color='white')
ax.legend(facecolor='#1a2235', edgecolor='#1e293b', labelcolor='white', fontsize=9)
ax.set_facecolor('#0f1828')
ax.tick_params(colors='white')
ax.spines[:].set_color('#1e293b')
ax.grid(alpha=0.15)
fig.patch.set_facecolor('#0a0e1a')

plt.tight_layout()
plt.savefig('dytopo_similarity_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

for r, (R, lbl) in enumerate(zip(code_R, LABELS)):
    vals = R[mask]
    above = (vals > TAU).sum()
    print(f'{lbl}: mean={vals.mean():.3f}, {above}/{len(vals)} pairs above τ → {above} directed edges')

## 5. t-SNE / PCA projection of all Q and K vectors

Visualise the 384-dim `all-MiniLM-L6-v2` embeddings in 2D to see why Round 1 Q/K cluster together (high cosine overlap) while Round 3 diverge.

In [ ]:
from sklearn.decomposition import PCA

# Collect all Q and K vectors with metadata
all_texts, all_labels, all_types, all_rounds = [], [], [], []
for r, (qs, ks) in enumerate(code_qk):
    for role, q, k in zip(CODE_ROLES, qs, ks):
        all_texts.append(q)
        all_labels.append(f'{role}\nQ')
        all_types.append('Q')
        all_rounds.append(r)
        all_texts.append(k)
        all_labels.append(f'{role}\nK')
        all_types.append('K')
        all_rounds.append(r)

embeddings = encode(all_texts)  # (n, 384)

pca = PCA(n_components=2)
proj = pca.fit_transform(embeddings)
var = pca.explained_variance_ratio_
print(f'PCA: PC1={var[0]:.1%}, PC2={var[1]:.1%} explained variance')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
ROUND_COLORS = ['#38bdf8', '#fbbf24', '#4ade80']
ROUND_NAMES  = ['Round 1 (Exploration)', 'Round 2 (Verification)', 'Round 3 (Finalization)']

for r, (ax, col, name) in enumerate(zip(axes, ROUND_COLORS, ROUND_NAMES)):
    mask_r = np.array(all_rounds) == r
    pts = proj[mask_r]
    types_r = np.array(all_types)[mask_r]
    labels_r = np.array(all_labels)[mask_r]

    # Q points (circles) and K points (triangles)
    for marker, vec_type in [('o', 'Q'), ('^', 'K')]:
        mask_t = types_r == vec_type
        ax.scatter(pts[mask_t, 0], pts[mask_t, 1],
                   c=col, marker=marker, s=120, alpha=0.85,
                   edgecolors='white', linewidths=0.5, zorder=3,
                   label=f'Q (query)' if vec_type == 'Q' else 'K (key)')

    # Draw Q→K lines per role to show match distance
    q_pts = pts[types_r == 'Q']
    k_pts = pts[types_r == 'K']
    for q, k in zip(q_pts, k_pts):
        ax.plot([q[0], k[0]], [q[1], k[1]], '-', color=col, alpha=0.3, linewidth=1.2)

    # Annotate roles
    for i, (pt, lbl) in enumerate(zip(pts, labels_r)):
        ax.annotate(lbl, pt, fontsize=6.5, ha='center', va='bottom',
                    color='white', xytext=(0, 5), textcoords='offset points')

    ax.set_title(name, fontsize=9, color='white', pad=8)
    ax.set_xlabel(f'PC1 ({var[0]:.1%})', fontsize=8, color='white')
    ax.set_ylabel(f'PC2 ({var[1]:.1%})', fontsize=8, color='white')
    ax.set_facecolor('#0f1828')
    ax.tick_params(colors='white', labelsize=7)
    ax.spines[:].set_color('#1e293b')
    ax.legend(fontsize=7, facecolor='#1a2235', edgecolor='#1e293b', labelcolor='white')
    ax.grid(alpha=0.12)

fig.patch.set_facecolor('#0a0e1a')
fig.suptitle('PCA Projection of all-MiniLM-L6-v2 Embeddings — Code Task Mock Q/K vectors\n'
             'Circles=Query, Triangles=Key, Lines=Q→K pair (shorter = more similar = more likely to create an edge)',
             fontsize=10, color='white', fontweight='bold')
plt.tight_layout()
plt.savefig('dytopo_embedding_pca.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved: dytopo_embedding_pca.png')

## 6. Summary — why mock data works

| Round | Q/K strategy | Mean cosine similarity | Edges at τ=0.3 | Pattern |
|-------|-------------|----------------------|----------------|--------|
| 1 (Exploration) | Broad, cross-role needs ("I need algorithm analysis", "I provide implementation") | High (~0.4-0.6) | Dense | All agents can coordinate |
| 2 (Verification) | Role-specific, targeted ("I need Tester to verify my code") | Medium (~0.2-0.4) | Moderate | Relevant pairs connect |
| 3 (Finalization) | Completion-signalling ("I need nothing further") | Low (~0.1-0.2) | Sparse | Minimal final communication |

**Key insight:** `all-MiniLM-L6-v2` is trained for semantic similarity tasks — so "algorithm analysis" and "algorithm requirements" 
are naturally close in embedding space, while "I need nothing further" is far from every other descriptor.
The mock data exploits this to create the Figure 4 evolution *without* running any LLM.

In [ ]:
# Final summary printout
print('=== Mock Data Summary ===')
print(f'Encoder: all-MiniLM-L6-v2 (384-dim, L2-normalised)')
print(f'Threshold τ=0.3 (optimal for code tasks, per paper Table 3)\n')

mask = ~np.eye(N, dtype=bool)
for r, (R, A, lbl) in enumerate(zip(code_R, code_A, LABELS)):
    off = R[mask]
    n_edges = int(A.sum())
    density = n_edges / (N * (N - 1))
    print(f'Round {r+1} ({lbl.split("(")[1].rstrip(")")}):')
    print(f'  Cosine sim: min={off.min():.3f}  mean={off.mean():.3f}  max={off.max():.3f}')
    print(f'  Edges above τ: {n_edges}/{N*(N-1)} ({density:.0%} density)')
    print(f'  Topology: {"DENSE" if density > 0.5 else "MODERATE" if density > 0.2 else "SPARSE"}')
    print()